In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import StackingRegressor


ModuleNotFoundError: No module named 'xgboost'

In [ ]:
df = pd.read_csv("./아마도최종데이터.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# 타겟 설정
target = "brent_close"  

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

X = df.drop(columns=[target, "Date", "uuid_list"])
y = df[target]

In [ ]:
lgbm = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    print(f"\n====== FOLD {fold} ======")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    lgbm.fit(X_train, y_train)
    pred = lgbm.predict(X_val)

    mae = mean_absolute_error(y_val, pred)
    rmse = np.sqrt(mean_squared_error(y_val, pred))

    print(f"MAE={mae:.4f}, RMSE={rmse:.4f}")


In [ ]:
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler

# 스케일링
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# (samples, timesteps=10, features)
def make_window(data, target, win=10):
    X_list, y_list = [], []
    for i in range(len(data) - win):
        X_list.append(data[i:i+win])
        y_list.append(target[i+win])
    return np.array(X_list), np.array(y_list)

X_lstm, y_lstm = make_window(X_scaled, y.values)

model = Sequential([
    LSTM(64, return_sequences=False, input_shape=(X_lstm.shape[1], X_lstm.shape[2])),
    Dense(1)
])
model.compile(optimizer='adam', loss='mae')
model.fit(X_lstm, y_lstm, epochs=20, batch_size=32)


In [ ]:
estimators = [
    ('lgbm', LGBMRegressor()),
    ('xgb', XGBRegressor()),
    ('rf', RandomForestRegressor())
]

stack = StackingRegressor(
    estimators=estimators,
    final_estimator=LinearRegression()
)

stack.fit(X, y)
pred = stack.predict(X)
print("Final Stacking RMSE:", np.sqrt(mean_squared_error(y, pred)))
